# Task 4: General Health Query Chatbot (Prompt Engineering Based)
**DevelopersHub Corporation — AI/ML Engineering Internship**

### Objective
Build a chatbot that answers general health-related questions using an LLM, with prompt engineering for friendly/clear responses and safety filters.

### Model Used
**`mistralai/Mistral-7B-Instruct-v0.2`** — exactly as suggested in the task sheet as the free alternative to OpenAI GPT-3.5.

⚠️ **IMPORTANT — for fastest run:**
1. In Colab, go to **Runtime → Change runtime type → T4 GPU** before running (Mistral-7B is too slow on CPU).
2. We load the model in **4-bit quantization** — this cuts memory use by ~75% and makes it load + run much faster, with almost no quality loss.
3. First run will take a few minutes to **download** the model (~4-5 GB in 4-bit). After that, generation itself is fast.

### No API key / No token needed
Mistral-7B-Instruct-v0.2 is a fully open model — no Hugging Face login or token required to download it.


## Step 1: Check GPU
Make sure you selected a GPU runtime (Runtime → Change runtime type → T4 GPU), otherwise this will be extremely slow.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "⚠️ No GPU detected! Go to Runtime > Change runtime type > T4 GPU, then re-run." 

## Step 2: Install Dependencies
`bitsandbytes` enables 4-bit quantized loading for speed.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece
print("✅ Libraries installed successfully!")

## Step 3: Load Mistral-7B-Instruct (4-bit, for speed)
This downloads the model the first time (cached after that). Loading in 4-bit makes it fit and run much faster on a free Colab GPU.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("⏳ Downloading & loading model (first run takes a few minutes)...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("✅ Model loaded successfully! Ready to chat.")

## Step 4: Prompt Engineering
Mistral-Instruct expects a specific chat format (`[INST] ... [/INST]`). We wrap every question in a "helpful health assistant" persona instruction.


In [ ]:
def build_prompt(user_query):
    system_instruction = (
        "Act like a helpful, friendly general health assistant. "
        "Answer the following health question in simple, clear language for a general audience. "
        "Do not give specific medical diagnoses, dosages, or treatment instructions. "
        "If the question requires a doctor's attention, gently suggest consulting a healthcare professional. "
        "Keep the answer short (3-5 sentences)."
    )
    prompt = f"<s>[INST] {system_instruction}\n\nQuestion: {user_query} [/INST]"
    return prompt

print("✅ Prompt template ready.")

## Step 5: Safety Filter
Keyword-based safety layer:
- Emergency queries → redirected to emergency services (model is NOT asked to answer these)
- Dosage-related queries → disclaimer added before the model's answer


In [ ]:
EMERGENCY_KEYWORDS = [
    "suicide", "kill myself", "overdose", "chest pain", "can't breathe",
    "cannot breathe", "severe bleeding", "unconscious", "heart attack", "stroke"
]

DOSAGE_KEYWORDS = ["how much", "dosage", "dose", "mg", "milligram", "how many pills", "how many tablets"]

def safety_check(user_query):
    q = user_query.lower()

    if any(word in q for word in EMERGENCY_KEYWORDS):
        return (
            "⚠️ This sounds like it could be a medical emergency. "
            "Please contact your local emergency services or go to the nearest emergency room immediately. "
            "I'm not able to provide emergency medical advice."
        )

    if any(word in q for word in DOSAGE_KEYWORDS):
        return "DOSAGE_WARNING"

    return None

print("✅ Safety filter ready.")

## Step 6: The Chatbot Function
Combines prompt engineering + safety filter + the model. `max_new_tokens` is kept small for speed.

In [ ]:
def ask_health_bot(user_query, max_new_tokens=120):
    safety_result = safety_check(user_query)
    if safety_result == "DOSAGE_WARNING":
        disclaimer = "⚠️ Note: I can't give exact dosage recommendations — please check the label or ask a pharmacist/doctor.\n\n"
    elif safety_result is not None:
        return safety_result
    else:
        disclaimer = ""

    prompt = build_prompt(user_query)

    output = chatbot(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id
    )

    full_text = output[0]['generated_text']
    # Extract only the model's reply (after [/INST])
    answer = full_text.split("[/INST]")[-1].strip()

    return disclaimer + answer

print("✅ Chatbot function ready! You can now ask it questions.")

## Step 7: Run the Example Queries from the Task Sheet

In [ ]:
example_queries = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "What should I do if I have a mild fever?",
    "How can I improve my sleep quality?"
]

for q in example_queries:
    print("🧑 User:", q)
    print("🤖 Bot  :", ask_health_bot(q))
    print("-" * 80)

## Step 8: Test the Safety Filter

In [ ]:
safety_test_queries = [
    "I have chest pain and can't breathe, what should I do?",
    "How many mg of ibuprofen should I take?"
]

for q in safety_test_queries:
    print("🧑 User:", q)
    print("🤖 Bot  :", ask_health_bot(q))
    print("-" * 80)

## Step 9: Interactive Chat (Optional)
Run this cell and type your own health questions. Type `exit` to stop.


In [ ]:
print("💬 Health Query Chatbot — type 'exit' to quit\n")
while True:
    user_input = input("You: ")
    if user_input.lower().strip() == "exit":
        print("Bot: Take care! 👋")
        break
    response = ask_health_bot(user_input)
    print("Bot:", response, "\n")

## Skills Demonstrated
- ✅ Prompt design and testing (Mistral instruct-format prompting, persona-based)
- ✅ Using APIs/LLMs locally — Hugging Face Mistral-7B-Instruct, no API key/token, 4-bit quantized for speed
- ✅ Safety handling in chatbot responses (emergency detection + dosage disclaimers)
- ✅ Building a simple conversational agent

## Notes for README.md (GitHub submission)
- **Task objective:** Build a chatbot that answers general health questions safely and clearly using an LLM.
- **Tool/Model used:** `mistralai/Mistral-7B-Instruct-v0.2` (Hugging Face, free, open-source, no API key) — loaded in 4-bit for speed on free Colab GPU.
- **Approach:** Prompt engineering (persona + instruction format `[INST]...[/INST]`) + keyword-based safety filter.
- **Key results:** Bot answers general health questions in plain language, redirects emergency-sounding queries to professional help, avoids giving exact dosages.

### Speed tip
If it's still too slow on your Colab GPU tier, you can swap `model_id` to `"mistralai/Mistral-7B-Instruct-v0.1"` or reduce `max_new_tokens` further (e.g., to 60) — no other code changes needed.
